In [33]:
import pandas as pd
import numpy as np
import ast
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix


In [20]:
# Load dataset
df = pd.read_csv("reviews.csv")

# Convert ratings column (string → dictionary)
df["ratings"] = df["ratings"].apply(ast.literal_eval)

# Extract overall rating
df["overall"] = df["ratings"].apply(lambda x: x.get("overall"))

# Keep required columns only
df = df[["text", "overall"]]

# Remove missing values
df = df.dropna()

print("Dataset shape:", df.shape)


Dataset shape: (878561, 2)


In [21]:
def convert_sentiment(rating):
    if rating <= 2:
        return 0   # Negative
    elif rating == 3:
        return 1   # Neutral
    else:
        return 2   # Positive

df["sentiment"] = df["overall"].apply(convert_sentiment)

print(df["sentiment"].value_counts())

sentiment
2    642046
1    122565
0    113950
Name: count, dtype: int64


In [22]:
X = df["text"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [39]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=20000,
        ngram_range=(1,2),
        stop_words="english",
        min_df=5
    )),
    ("model", LogisticRegression(
        max_iter=500,
        class_weight="balanced",
        solver="lbfgs",
        multi_class="multinomial"
    ))
])


In [40]:
pipeline.fit(X_train, y_train)


MemoryError: 

In [25]:
y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.8049148327101581

Classification Report:

              precision    recall  f1-score   support

           0       0.78      0.72      0.75     22790
           1       0.41      0.69      0.51     24513
           2       0.96      0.84      0.90    128410

    accuracy                           0.80    175713
   macro avg       0.71      0.75      0.72    175713
weighted avg       0.86      0.80      0.82    175713



In [18]:
joblib.dump(pipeline, "hotel_sentiment_model.pkl")

['hotel_sentiment_model.pkl']

In [41]:
model = joblib.load("hotel_sentiment_model.pkl")

In [44]:
review = "it was agreat stay, the staff was friendly and the room was clean and comfortable"

prediction = model.predict([review])[0]

label_map = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

print("Sentiment:", label_map[prediction])

Sentiment: Positive
